# Auditable Recursive Latent Reasoner (RLR)

This notebook is a **reproducible, fail-closed implementation audit**, not a marketing benchmark. It implements an explicit recurrent state-update mechanism for function-composition tasks, validates it against independently computed ground truth, logs every evaluated example, and produces checksumed artifacts.

## What it proves

- The implementation recursively applies one transformation per loop.
- It solves held-out compositions longer than those used in the calibration suite.
- Removing loops or replacing transformations with frozen/random controls reduces accuracy.

## What it does not prove

It does **not** prove that an LLM has emergent reasoning, nor that Mamba-2 is superior to GRU-RSSM. It is a controlled mechanistic RLR experiment. Those stronger claims require a separately trained model, fair baselines, and independent held-out evaluation.

Run all cells from a new Colab runtime. Do not edit generated artifacts after the run.

In [1]:
# Environment and immutable run configuration
import sys, os, json, time, random, hashlib, platform, subprocess, zipfile
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import torch

RUN_ID = datetime.now(timezone.utc).strftime('rlr_%Y%m%dT%H%M%SZ')
OUT = Path('/content/drive/MyDrive/RLR_Auditable_Runs') / RUN_ID

# Set True in Colab to persist all raw artifacts.
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
OUT.mkdir(parents=True, exist_ok=False)

CONFIG = {
    'run_id': RUN_ID,
    'task': 'random_permutation_function_composition',
    'vocabulary_size': 32,
    'calibration_hops': [1, 2, 3, 4, 5],
    'held_out_hops': [6, 7, 8, 9, 10, 11, 12],
    'examples_per_hop_per_seed': 250,
    'seeds': [101, 202, 303, 404, 505],
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'controls': ['full_recursive', 'max_loops_1', 'frozen_random_transition'],
    'code_version': 'auditable-rlr-v1',
}

(OUT / 'config.json').write_text(json.dumps(CONFIG, indent=2, sort_keys=True))
print('Run ID:', RUN_ID)
print('Artifact directory:', OUT)
print('Device:', CONFIG['device'])
print('Torch:', torch.__version__, '| CUDA:', torch.version.cuda)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


Mounted at /content/drive
Run ID: rlr_20260914T094741Z
Artifact directory: /content/drive/MyDrive/RLR_Auditable_Runs/rlr_20260914T094741Z
Device: cpu
Torch: 2.11.0+cpu | CUDA: None


## Task and RLR implementation

Each example supplies an ordered list of randomly generated bijective functions over a 32-symbol vocabulary and one initial symbol. The correct answer is obtained by applying each function in order. The RLR state is a soft one-hot distribution over symbols. Each recurrent loop multiplies that state by exactly one supplied transition matrix.

This design intentionally makes the causal role of recurrence inspectable: a run with fewer loops cannot generally compute a longer composition. The evaluator has a separately implemented NumPy oracle; it does not trust the PyTorch model output.

In [2]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def make_permutation(rng, vocab_size):
    return rng.permutation(vocab_size).astype(np.int64)

def compose_oracle(permutations, start):
    value = int(start)
    trace = [value]
    for perm in permutations:
        value = int(perm[value])
        trace.append(value)
    return value, trace

def permutation_to_matrix(perm, vocab_size):
    matrix = np.zeros((vocab_size, vocab_size), dtype=np.float32)
    matrix[np.arange(vocab_size), perm] = 1.0
    return matrix

def generate_example(rng, hops, vocab_size):
    permutations = [make_permutation(rng, vocab_size) for _ in range(hops)]
    start = int(rng.integers(vocab_size))
    target, trace = compose_oracle(permutations, start)
    return {
        'start': start,
        'target': target,
        'trace': trace,
        'permutations': [p.tolist() for p in permutations],
        'matrices': np.stack([permutation_to_matrix(p, vocab_size) for p in permutations]),
    }

class ExplicitRLR(torch.nn.Module):
    """One reusable transition operator applied recursively to a soft symbolic state."""
    def __init__(self, vocab_size):
        super().__init__()
        self.vocab_size = vocab_size

    @torch.no_grad()
    def forward(self, matrices, starts, max_loops=None, replacement_matrices=None):
        # matrices: [batch, hops, vocab, vocab], row i maps source i to destination.
        batch, hops, vocab, _ = matrices.shape
        if vocab != self.vocab_size:
            raise ValueError('Vocabulary mismatch')
        loops = hops if max_loops is None else min(int(max_loops), hops)
        active = matrices if replacement_matrices is None else replacement_matrices
        state = torch.nn.functional.one_hot(starts, num_classes=vocab).float()
        states = [state.detach().cpu()]
        for step in range(loops):
            state = torch.bmm(state.unsqueeze(1), active[:, step]).squeeze(1)
            states.append(state.detach().cpu())
        return state, states

def make_random_replacement(rng, hops, vocab_size):
    return np.stack([permutation_to_matrix(make_permutation(rng, vocab_size), vocab_size) for _ in range(hops)])


In [3]:
# Evaluator: generates held-out examples, evaluates each control, and writes raw JSONL.
def evaluate_condition(model, example, condition, seed, example_id, device):
    vocab_size = model.vocab_size
    hops = len(example['permutations'])
    matrices_np = example['matrices'][None, ...]
    starts = torch.tensor([example['start']], dtype=torch.long, device=device)
    matrices = torch.tensor(matrices_np, dtype=torch.float32, device=device)

    if condition == 'full_recursive':
        state, states = model(matrices, starts, max_loops=hops)
        loops_used = hops
    elif condition == 'max_loops_1':
        state, states = model(matrices, starts, max_loops=1)
        loops_used = 1
    elif condition == 'frozen_random_transition':
        control_rng = np.random.default_rng(seed * 1_000_003 + example_id * 97 + hops)
        replacement = torch.tensor(
            make_random_replacement(control_rng, hops, vocab_size)[None, ...],
            dtype=torch.float32, device=device
        )
        state, states = model(matrices, starts, max_loops=hops, replacement_matrices=replacement)
        loops_used = hops
    else:
        raise ValueError(condition)

    prediction = int(state.argmax(dim=-1).item())
    return {
        'run_id': RUN_ID,
        'seed': seed,
        'example_id': example_id,
        'condition': condition,
        'hop_count': hops,
        'start': example['start'],
        'target': example['target'],
        'prediction': prediction,
        'correct': bool(prediction == example['target']),
        'loops_used': loops_used,
        'oracle_trace': example['trace'],
        'permutations': example['permutations'],
        'final_state_argmax': prediction,
        'final_state_probability': float(state.max().item()),
        'state_trace_argmax': [int(s.argmax(dim=-1).item()) for s in states],
    }

model = ExplicitRLR(CONFIG['vocabulary_size']).to(CONFIG['device']).eval()
raw_path = OUT / 'raw_results.jsonl'
records = []

with raw_path.open('w') as f:
    for seed in CONFIG['seeds']:
        set_seed(seed)
        rng = np.random.default_rng(seed)
        example_id = 0
        for hops in CONFIG['held_out_hops']:
            for _ in range(CONFIG['examples_per_hop_per_seed']):
                example = generate_example(rng, hops, CONFIG['vocabulary_size'])
                for condition in CONFIG['controls']:
                    record = evaluate_condition(
                        model, example, condition, seed, example_id, CONFIG['device']
                    )
                    f.write(json.dumps(record, separators=(',', ':')) + '\n')
                    records.append(record)
                example_id += 1

print('Raw records:', len(records))
print('Raw artifact:', raw_path)
assert len(records) == (len(CONFIG['seeds']) * len(CONFIG['held_out_hops']) *
                        CONFIG['examples_per_hop_per_seed'] * len(CONFIG['controls']))


Raw records: 26250
Raw artifact: /content/drive/MyDrive/RLR_Auditable_Runs/rlr_20260914T094741Z/raw_results.jsonl


In [4]:
# Independent aggregation and fail-closed integrity checks.
from collections import defaultdict

def wilson_interval(successes, total, z=1.96):
    if total == 0:
        return (float('nan'), float('nan'))
    p = successes / total
    denom = 1 + z*z/total
    centre = (p + z*z/(2*total)) / denom
    margin = z * ((p*(1-p)/total + z*z/(4*total*total)) ** 0.5) / denom
    return max(0.0, centre-margin), min(1.0, centre+margin)

summary = {}
for condition in CONFIG['controls']:
    for hops in CONFIG['held_out_hops']:
        subset = [r for r in records if r['condition'] == condition and r['hop_count'] == hops]
        n = len(subset)
        k = sum(r['correct'] for r in subset)
        lo, hi = wilson_interval(k, n)
        summary[f'{condition}|hops={hops}'] = {
            'condition': condition, 'hop_count': hops, 'n': n, 'correct': k,
            'accuracy': k / n, 'wilson_95_low': lo, 'wilson_95_high': hi,
        }

for condition in CONFIG['controls']:
    subset = [r for r in records if r['condition'] == condition]
    n = len(subset)
    k = sum(r['correct'] for r in subset)
    lo, hi = wilson_interval(k, n)
    summary[f'{condition}|ALL'] = {
        'condition': condition, 'hop_count': 'ALL', 'n': n, 'correct': k,
        'accuracy': k / n, 'wilson_95_low': lo, 'wilson_95_high': hi,
    }

# Integrity condition 1: every FULL state trace must equal the separately computed oracle trace.
full = [r for r in records if r['condition'] == 'full_recursive']
trace_mismatches = [r for r in full if r['state_trace_argmax'] != r['oracle_trace']]
if trace_mismatches:
    raise RuntimeError(f'FAIL: {len(trace_mismatches)} full recursive traces differ from oracle.')

# Integrity condition 2: no full result may be incorrect. This is an implementation proof, not a soft threshold.
full_errors = [r for r in full if not r['correct']]
if full_errors:
    raise RuntimeError(f'FAIL: full recursive implementation made {len(full_errors)} errors.')

# Integrity condition 3: controls must not accidentally equal FULL on every example.
for control in ['max_loops_1', 'frozen_random_transition']:
    control_acc = summary[f'{control}|ALL']['accuracy']
    if control_acc >= 0.95:
        raise RuntimeError(f'FAIL: control {control} unexpectedly has {control_acc:.3f} accuracy.')

summary_path = OUT / 'summary.json'
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps({k: v for k, v in summary.items() if k.endswith('|ALL')}, indent=2))
print('PASS: oracle traces match every full-recursive trace')


{
  "full_recursive|ALL": {
    "condition": "full_recursive",
    "hop_count": "ALL",
    "n": 8750,
    "correct": 8750,
    "accuracy": 1.0,
    "wilson_95_low": 0.999561152671531,
    "wilson_95_high": 0.9999999999999999
  },
  "max_loops_1|ALL": {
    "condition": "max_loops_1",
    "hop_count": "ALL",
    "n": 8750,
    "correct": 287,
    "accuracy": 0.0328,
    "wilson_95_low": 0.029268172510689736,
    "wilson_95_high": 0.03674188643303161
  },
  "frozen_random_transition|ALL": {
    "condition": "frozen_random_transition",
    "hop_count": "ALL",
    "n": 8750,
    "correct": 272,
    "accuracy": 0.031085714285714286,
    "wilson_95_low": 0.027650045649861118,
    "wilson_95_high": 0.03493294648470069
  }
}
PASS: oracle traces match every full-recursive trace


In [5]:
# Provenance manifest, human-readable report, and immutable archive.
def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

try:
    git_head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
except Exception:
    git_head = None

manifest = {
    'run_id': RUN_ID,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'config': CONFIG,
    'python': sys.version,
    'platform': platform.platform(),
    'torch_version': torch.__version__,
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'git_head_if_available': git_head,
    'integrity': {
        'oracle_trace_mismatches': len(trace_mismatches),
        'full_recursive_errors': len(full_errors),
        'status': 'PASS',
    },
    'files_before_manifest': {
        p.name: file_hash(p) for p in OUT.iterdir() if p.is_file()
    },
}
manifest_path = OUT / 'manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))

rows = [summary[f'{condition}|ALL'] for condition in CONFIG['controls']]
report = [
    '# Auditable RLR Report',
    '',
    f'- Run ID: `{RUN_ID}`',
    f'- Device: `{CONFIG["device"]}`',
    f'- Vocabulary: {CONFIG["vocabulary_size"]}',
    f'- Held-out chain lengths: {CONFIG["held_out_hops"]}',
    f'- Independent seeds: {len(CONFIG["seeds"])}',
    f'- Examples per hop per seed: {CONFIG["examples_per_hop_per_seed"]}',
    '',
    '## Result summary',
    '',
    '| Condition | Correct / N | Accuracy | Wilson 95% CI |',
    '|---|---:|---:|---:|',
]
for row in rows:
    report.append(
        f'| {row["condition"]} | {row["correct"]} / {row["n"]} | '
        f'{row["accuracy"]:.4f} | [{row["wilson_95_low"]:.4f}, {row["wilson_95_high"]:.4f}] |'
    )
report += [
    '',
    '## Integrity result',
    '',
    '- Every full-recursive intermediate state exactly matched the independent NumPy oracle trace.',
    '- The implementation aborts if any full-recursive prediction is wrong.',
    '- The ablations use the same examples but either only one update or random replacement transitions.',
    '',
    '## Correct interpretation',
    '',
    'This is evidence that this explicit recurrent program performs compositional state updates and needs its loops for this task.',
    'It is not evidence that a language model has general reasoning ability, and it makes no Mamba-2 or RL superiority claim.',
]
report_path = OUT / 'REPORT.md'
report_path.write_text('\n'.join(report) + '\n')

archive_path = OUT.parent / f'{RUN_ID}_artifacts.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():
        if p.is_file():
            z.write(p, arcname=f'{RUN_ID}/{p.name}')

print(report_path.read_text())
print('Manifest:', manifest_path)
print('Archive:', archive_path)
print('Archive SHA256:', file_hash(archive_path))


# Auditable RLR Report

- Run ID: `rlr_20260914T094741Z`
- Device: `cpu`
- Vocabulary: 32
- Held-out chain lengths: [6, 7, 8, 9, 10, 11, 12]
- Independent seeds: 5
- Examples per hop per seed: 250

## Result summary

| Condition | Correct / N | Accuracy | Wilson 95% CI |
|---|---:|---:|---:|
| full_recursive | 8750 / 8750 | 1.0000 | [0.9996, 1.0000] |
| max_loops_1 | 287 / 8750 | 0.0328 | [0.0293, 0.0367] |
| frozen_random_transition | 272 / 8750 | 0.0311 | [0.0277, 0.0349] |

## Integrity result

- Every full-recursive intermediate state exactly matched the independent NumPy oracle trace.
- The implementation aborts if any full-recursive prediction is wrong.
- The ablations use the same examples but either only one update or random replacement transitions.

## Correct interpretation

This is evidence that this explicit recurrent program performs compositional state updates and needs its loops for this task.
It is not evidence that a language model has general reasoning ability, and it

## Next research step

Use this notebook as an **integrity template** before adding a learned neural RLR: retain the independent oracle, per-example JSONL logging, frozen controls, explicit no-loop control, held-out generator seeds, checksums, and fail-closed assertions.

For a learned-model claim, add a training split that is disjoint from the held-out generator, train a neural transition model without direct access to the oracle matrices, compare it with parameter- and compute-matched baselines over at least five independent training seeds, and report the raw learning curves and uncertainty intervals.